In [ ]:
# Ekstrak Landmark untuk Huruf
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import random

# ================================
# PATH DATASET
# ================================
DATASET_PATH = "alfabet_bisindo"
IMAGE_ROOT = os.path.join(DATASET_PATH, "images")
SPLITS = ["train", "val"]
OUTPUT_CSV = "bisindo_alfabet_final.csv"

# ================================
# MEDIAPIPE HANDS
# ================================
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.6
)

# ================================
# AUGMENTASI
# ================================
def augment_image(img):
    h, w = img.shape[:2]
    aug = img.copy()

    alpha = random.uniform(0.9, 1.1)
    beta = random.randint(-15, 15)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    return aug

# ================================
# EKSTRAK 1 TANGAN
# ================================
def extract_one_hand(hand_landmarks):
    lm = hand_landmarks.landmark
    pts = np.array([[p.x, p.y] for p in lm])

    base = pts[0]
    pts = pts - base

    max_val = np.max(np.abs(pts))
    if max_val != 0:
        pts = pts / max_val

    data = pts.flatten().tolist()

    def dist(a, b):
        return ((lm[a].x - lm[b].x) ** 2 + (lm[a].y - lm[b].y) ** 2) ** 0.5

    data.append(dist(4, 8))
    data.append(dist(8, 12))
    data.append(dist(12, 16))
    data.append(dist(16, 20))

    return data

# ================================
# EKSTRAK 2 TANGAN
# ================================
def extract_landmarks(rgb_img):
    result = hands.process(rgb_img)

    if not result.multi_hand_landmarks:
        return None

    empty_hand = [0.0] * 46
    left_data = empty_hand.copy()
    right_data = empty_hand.copy()

    for hand_landmarks, handedness in zip(
        result.multi_hand_landmarks,
        result.multi_handedness
    ):
        hand_label = handedness.classification[0].label

        if hand_label == "Left":
            left_data = extract_one_hand(hand_landmarks)
        elif hand_label == "Right":
            right_data = extract_one_hand(hand_landmarks)

    data = left_data + right_data

    if len(data) != 92:
        return None

    return data

# ================================
# PROSES DATASET
# ================================
rows = []
label_counts = {}

for split in SPLITS:
    split_path = os.path.join(IMAGE_ROOT, split)

    if not os.path.exists(split_path):
        print("Folder tidak ditemukan:", split_path)
        continue

    print(f"\nMemproses split: {split}")

    for label in sorted(os.listdir(split_path)):
        class_folder = os.path.join(split_path, label)

        if not os.path.isdir(class_folder):
            continue

        print(f"Class: {label}")

        for img_file in os.listdir(class_folder):
            if not img_file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue

            img_path = os.path.join(class_folder, img_file)
            img = cv2.imread(img_path)

            if img is None:
                continue

            img = cv2.resize(img, (640, 480))

            # gambar asli
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            data = extract_landmarks(rgb)

            if data:
                rows.append(data + [label])
                label_counts[label] = label_counts.get(label, 0) + 1

            # gambar augmentasi
            aug = augment_image(img)
            rgb_aug = cv2.cvtColor(aug, cv2.COLOR_BGR2RGB)
            data_aug = extract_landmarks(rgb_aug)

            if data_aug:
                rows.append(data_aug + [label])
                label_counts[label] = label_counts.get(label, 0) + 1

# ================================
# SIMPAN CSV
# ================================
header = [f"f{i}" for i in range(92)] + ["label"]

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

hands.close()

print("\n✅ selesai:", OUTPUT_CSV)
print("Total data:", len(rows))
print("Distribusi label:")
for label, count in sorted(label_counts.items()):
    print(label, ":", count)